# Analisi 3 — visualizzazione patch
Coord in frame **20x**, slide **40x** a livello0 → mapping `coord*factor` (factor=AppMag/20, auto-calibrato). Coord dal parquet, slide in `../gdc/`.

## 0. Config

In [ ]:
import os, glob, json as _json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle
from PIL import Image
import openslide
import pyarrow.parquet as pq, pyarrow.compute as pc

OUT_DIR   = '../grad/xai_outputs'
DL_DIR    = '../gdc'
FIG_DIR   = os.path.join(OUT_DIR, 'patch_figs'); os.makedirs(FIG_DIR, exist_ok=True)
EXPLAIN_TOP3 = '../coad_stad_explainability/explain_top3.parquet'
META_JSON = '../metadata.repository.2026-06-02.json'

PATCH_20X, DISPLAY, THUMB_MAX = 224, 512, 1400
CONTEXT, N_SHOW, FACTOR_CAND = 3, 6, [2, 1]
COMBINED_ABS = True
K_OVERLAY = [10, 20, 50, 100]
OVERLAY_SCORE = 'combined'      # 'grad_score' (mostra le uniche) | 'combined'
COMPARE       = 'codominant'      # 'codominant' (come Jaccard paper) | 'all'

img   = pd.read_csv(os.path.join(OUT_DIR, 'immagini_da_scaricare.csv'))
patch = pd.read_csv(os.path.join(OUT_DIR, 'patches_da_guardare.csv'))
print('slide:', len(img), '| patch:', len(patch))

## 1. Helper

In [ ]:
def find_svs(slide):
    h = glob.glob(os.path.join(DL_DIR, '*', slide+'.svs')) + glob.glob(os.path.join(DL_DIR, slide+'.svs'))
    return h[0] if h else None

def slide_mag(wsi):
    for k in ('aperio.AppMag', openslide.PROPERTY_NAME_OBJECTIVE_POWER):
        v = wsi.properties.get(k)
        if v:
            try: return float(v)
            except ValueError: pass
    return None

def read_win(wsi, x, y, factor, context):
    p0 = PATCH_20X*factor; win = p0*context; W, H = wsi.dimensions
    cx, cy = x*factor, y*factor
    ox = int(np.clip(cx-(win-p0)//2, 0, max(0, W-win))); oy = int(np.clip(cy-(win-p0)//2, 0, max(0, H-win)))
    im = wsi.read_region((ox, oy), 0, (win, win)).convert('RGB').resize((DISPLAY, DISPLAY), Image.LANCZOS)
    return im, ((cx-ox)/win*DISPLAY, (cy-oy)/win*DISPLAY, p0/win*DISPLAY)

def _white(im): return float((np.asarray(im.convert('L'))>220).mean())

def calibrate_factor(wsi, tp, mag):
    W, H = wsi.dimensions
    f0 = max(1, int(round(mag/20))) if mag else None
    cand = ([f0] + [f for f in FACTOR_CAND if f != f0]) if f0 else FACTOR_CAND
    sc = {}
    for f in cand:
        wf, n = 0.0, 0
        for _, pr in tp.head(6).iterrows():
            x, y = int(pr['coord_x']), int(pr['coord_y'])
            if x*f+PATCH_20X*f > W or y*f+PATCH_20X*f > H: wf, n = 1.0, 1; break
            wf += _white(read_win(wsi, x, y, f, 1)[0]); n += 1
        sc[f] = wf/max(n, 1)
    return min(sc, key=sc.get), sc

def load_attr(tumor, case, slide=None):
    path = os.path.join(OUT_DIR, f'attr_{tumor.lower()}_subset_scores.parquet')
    if not os.path.exists(path): print('  cache mancante:', os.path.basename(path)); return None
    d = pq.read_table(path, columns=['sample_id','case_id','signature_name','coord_x','coord_y',
        'original_index','attention_weight','grad_score'], filters=[('case_id','==',case)]).to_pandas()
    if d.empty: return None
    if slide and slide in set(d['sample_id'].astype(str)): d = d[d['sample_id'].astype(str)==slide]
    d['signature_name'] = d['signature_name'].astype(str)
    g = d['grad_score'].astype('float32')
    d['combined'] = d['attention_weight'].astype('float32') * (np.abs(g) if COMBINED_ABS else np.clip(g, 0, None))
    d['grad_score'] = np.abs(g)
    return d

_EXPL = {}
def codominant(case):
    _EXPL.setdefault('df', pq.read_table(EXPLAIN_TOP3).to_pandas())
    e = _EXPL['df']
    return set(e[(e['Case ID']==case) & (e['group']=='plus')]['signature'].astype(str))

def topk(d, sig, K, score=None):
    col = 'combined' if (score or OVERLAY_SCORE)=='combined' else 'grad_score'
    return d[d['signature_name']==sig].nlargest(K, col)

def others_of(d, sig, case):
    sigs = [s for s in d['signature_name'].unique() if s != sig]
    return [s for s in sigs if s in codominant(case)] if COMPARE=='codominant' else sigs

def show_slide(row, n_show=N_SHOW, force_factor=None):
    slide, tumor, sig = row['slide'], row['tumor'], row['signature']
    svs = find_svs(slide); tp = patch[patch['slide']==slide].sort_values('rank').head(n_show)
    if svs is None: print(f'[{tumor} {sig}] .svs non trovata -> skip'); return
    wsi = openslide.open_slide(svs); W, H = wsi.dimensions; mag = slide_mag(wsi)
    factor, sc = (force_factor, {}) if force_factor else calibrate_factor(wsi, tp, mag)
    print(f'[{tumor} {sig}] {slide[:24]} L0={W}x{H} AppMag={mag} factor={factor} white={ {k:round(v,2) for k,v in sc.items()} }')
    thumb = wsi.get_thumbnail((THUMB_MAX, int(THUMB_MAX*H/W))); sx, sy = thumb.width/W, thumb.height/H
    ncol = 3; nrow = int(np.ceil(len(tp)/ncol))
    fig = plt.figure(figsize=(6+2.4*ncol, max(4.5, 2.4*nrow)))
    gs = gridspec.GridSpec(max(nrow, 2), 2+ncol, figure=fig, wspace=0.15, hspace=0.25)
    axt = fig.add_subplot(gs[:, :2]); axt.imshow(thumb); axt.axis('off')
    for k, (_, pr) in enumerate(tp.iterrows(), 1):
        bx, by, bs = pr['coord_x']*factor*sx, pr['coord_y']*factor*sy, PATCH_20X*factor*sx
        axt.add_patch(Rectangle((bx, by), bs, bs, fill=False, edgecolor='red', linewidth=1.5))
        axt.text(bx, by-2, str(k), color='red', fontsize=9, fontweight='bold', va='bottom')
    axt.set_title(f'{tumor} — {sig} (factor={factor})\n{slide[:30]}...', fontsize=10)
    for k, (_, pr) in enumerate(tp.iterrows()):
        ax = fig.add_subplot(gs[k//ncol, 2+k%ncol])
        im, _ = read_win(wsi, int(pr['coord_x']), int(pr['coord_y']), factor, CONTEXT)
        ax.imshow(im); ax.axis('off'); ax.set_title(f'#{k+1}  {pr["combined_score"]:.2e}', fontsize=8)
    wsi.close(); plt.savefig(os.path.join(FIG_DIR, f'{tumor}_{sig}_{slide[:20]}.png'), dpi=150, bbox_inches='tight'); plt.show()

def overlay_slide(row, Ks=K_OVERLAY, force_factor=None):
    slide, tumor, sig, case = row['slide'], row['tumor'], row['signature'], row['case_id']
    svs = find_svs(slide)
    if svs is None: print(f'[{tumor} {sig}] .svs non trovata -> skip'); return
    d = load_attr(tumor, case, slide)
    if d is None or d.empty: print(f'[{tumor} {sig}] no cache per {case} -> skip'); return
    wsi = openslide.open_slide(svs); W, H = wsi.dimensions; mag = slide_mag(wsi)
    factor, _ = (force_factor, {}) if force_factor else calibrate_factor(wsi, topk(d, sig, 6), mag)
    thumb = wsi.get_thumbnail((THUMB_MAX, int(THUMB_MAX*H/W))); sx, sy = thumb.width/W, thumb.height/H
    others = others_of(d, sig, case)
    if not others: print(f'  [{tumor} {sig}] nessuna co-dominante: tutte uniche')
    fig, axes = plt.subplots(1, len(Ks), figsize=(4.3*len(Ks), 4.6), squeeze=False)
    for j, K in enumerate(Ks):
        ax = axes[0][j]; ax.imshow(thumb); ax.axis('off')
        st = topk(d, sig, K); ids = set(st['original_index'])
        un = set().union(*[set(topk(d, l, K)['original_index']) for l in others]) if others else set()
        uniq = ~st['original_index'].isin(un)
        for mask, col, lab in [(uniq, '#1f77b4', 'unique'), (~uniq, '#ff7f0e', 'shared')]:
            s2 = st[mask]; ax.scatter(s2['coord_x']*factor*sx, s2['coord_y']*factor*sy, s=20, c=col, edgecolor='k', linewidth=0.3, label=lab, zorder=3)
        J = len(ids & un)/len(ids | un) if (ids | un) else 0.0
        ax.set_title(f'K={K}  U={float(uniq.mean()):.2f}  J={J:.2f}', fontsize=9)
        if j == len(Ks)-1: ax.legend(loc='lower right', fontsize=8, framealpha=0.9, markerscale=1.4)
    fig.suptitle(f'{tumor} — {sig} ({slide[:26]}) [{OVERLAY_SCORE}, vs {COMPARE}]', y=1.02, fontsize=11)
    wsi.close(); plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'overlay_{tumor}_{sig}_{slide[:20]}.png'), dpi=150, bbox_inches='tight'); plt.show()

## 2. Tutte le slide (patch laterali)

In [ ]:
for _, row in img.iterrows():
    show_slide(row)

## 3. Focus / forzare factor

In [ ]:
SEL = 0
show_slide(img.iloc[SEL], n_show=12, force_factor=2)   # force_factor=1 se resta bianca

## 4. Overlay top-K: unique (blu) vs shared (arancio)

In [ ]:
for _, row in img.iterrows():
    overlay_slide(row)

## 5. TP vs TN affiancate — esemplari (firme note + una non nota)

In [ ]:
PAIR_SCORE, PAIR_K = 'combined', 50      # 'combined' (paper) | 'grad_score' (piu' uniche visibili)
CROP_CONTEXT, N_UNIQUE, N_SHARED, N_CAND = 1, 2, 2, 8

def fileid_map():
    return {r['file_name'].replace('.svs',''): r['file_id'] for r in _json.load(open(META_JSON))
            if 'file_name' in r and 'file_id' in r}

def pair_candidates(tumor, sig, n=N_CAND):
    """(tp_candidates, tn_candidates): per role, cached cases with slide/file_id/downloaded."""
    path = os.path.join(OUT_DIR, f'attr_{tumor.lower()}_subset_scores.parquet')
    if not os.path.exists(path): return [], []
    e = pq.read_table(EXPLAIN_TOP3).to_pandas()
    es = e[(e['tumor_type'].str.upper()==tumor.upper()) & (e['signature']==sig)]
    stt = {r['Case ID']: ('TP' if (r['group']=='plus' and r['eligible']=='yes')
           else 'TN' if (r['group']=='minus' and r['eligible']=='no') else 'x') for _, r in es.iterrows()}
    cache = set(pc.unique(pq.read_table(path, columns=['case_id']).column('case_id')).to_pylist())
    fid = fileid_map()
    def cands(cases, role):
        out = []; seen = 0
        for c in cases:
            d = load_attr(tumor, c)
            if d is None: continue
            seen += 1
            out += [{'role':role,'case':c,'slide':s,'fid':fid.get(s,'?'),'dl':find_svs(s) is not None,'d':d}
                    for s in sorted(d['sample_id'].astype(str).unique())]
            if any(o['dl'] for o in out) or seen >= n: break
        return out
    return (cands([c for c,s in stt.items() if s=='TP' and c in cache], 'TP'),
            cands([c for c,s in stt.items() if s=='TN' and c in cache], 'TN'))

def row_data(tumor, sig, r, K, score):
    slide = r['slide']; d = r['d'][r['d']['sample_id'].astype(str)==slide]
    wsi = openslide.open_slide(find_svs(slide)); W, H = wsi.dimensions
    factor, _ = calibrate_factor(wsi, topk(d, sig, 6, score), slide_mag(wsi))
    thumb = wsi.get_thumbnail((THUMB_MAX, int(THUMB_MAX*H/W))); sx, sy = thumb.width/W, thumb.height/H
    others = others_of(d, sig, r['case'])
    st = topk(d, sig, K, score)
    un = set().union(*[set(topk(d, l, K, score)['original_index']) for l in others]) if others else set()
    uniq = ~st['original_index'].isin(un)
    us = st[uniq].nlargest(N_UNIQUE, 'combined'); ss = st[~uniq].nlargest(N_SHARED, 'combined')
    sel = pd.concat([us, ss]).assign(kind=['unique']*len(us)+['shared']*len(ss))
    return wsi, thumb, sx, sy, factor, st, uniq, sel

def paired_tp_tn(tumor, sig, K=PAIR_K, score=PAIR_SCORE):
    tp_c, tn_c = pair_candidates(tumor, sig)
    pick = lambda cs: next((o for o in cs if o['dl']), None)
    tp, tn = pick(tp_c), pick(tn_c)
    if tp is None or tn is None:
        print(f'[{tumor}/{sig}] manca {"TP" if tp is None else ""}{" e " if tp is None and tn is None else ""}{"TN" if tn is None else ""} scaricata -> usa download_plan'); return
    fig = plt.figure(figsize=(15, 9))
    outer = gridspec.GridSpec(2, 2, width_ratios=[1.5, 1], hspace=0.18, wspace=0.06, figure=fig)
    for ri, (tag, r) in enumerate([('TP', tp), ('TN', tn)]):
        wsi, thumb, sx, sy, factor, st, uniq, sel = row_data(tumor, sig, r, K, score)
        axt = fig.add_subplot(outer[ri, 0]); axt.imshow(thumb); axt.axis('off')
        for mask, col in [(uniq, '#1f77b4'), (~uniq, '#ff7f0e')]:
            s2 = st[mask]; axt.scatter(s2['coord_x']*factor*sx, s2['coord_y']*factor*sy, s=16, c=col, edgecolor='k', linewidth=0.3, zorder=3)
        for m, (_, pr) in enumerate(sel.iterrows(), 1):
            x, y = pr['coord_x']*factor*sx, pr['coord_y']*factor*sy
            axt.scatter([x], [y], s=120, facecolors='none', edgecolors='k', linewidths=1.6, zorder=4)
            axt.text(x, y, str(m), color='k', fontsize=9, fontweight='bold', ha='center', va='center', zorder=5)
        axt.set_title(f'{tag}  {r["case"]}  U={float(uniq.mean()):.2f} (K={K})', fontsize=12)
        inner = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=outer[ri, 1], wspace=0.08, hspace=0.22)
        for m, (_, pr) in enumerate(sel.iterrows()):
            axc = fig.add_subplot(inner[m//2, m%2]); im, _ = read_win(wsi, int(pr['coord_x']), int(pr['coord_y']), factor, CROP_CONTEXT)
            axc.imshow(im); axc.set_xticks([]); axc.set_yticks([])
            col = '#1f77b4' if pr['kind']=='unique' else '#ff7f0e'
            for sp in axc.spines.values(): sp.set_edgecolor(col); sp.set_linewidth(3)
            axc.set_title(f'#{m+1} {pr["kind"]}', fontsize=10, color=col)
        wsi.close()
    fig.legend(handles=[plt.Line2D([],[],marker='o',ls='',color='#1f77b4',label='unique',markersize=9),
                        plt.Line2D([],[],marker='o',ls='',color='#ff7f0e',label='shared',markersize=9)], loc='upper right', fontsize=10, frameon=False)
    fig.suptitle(f'{tumor} — {sig} (K={K}, {score}, vs co-dominant): TP vs TN', y=0.995, fontsize=13)
    plt.savefig(os.path.join(FIG_DIR, f'paired_TPvsTN_{tumor}_{sig}_{score}_K{K}.png'), dpi=200, bbox_inches='tight'); plt.show()

def download_plan(pairs, alt=2, ids_file='../gdc/ids_paired.txt'):
    """For each pair, collect the missing TP/TN slides (with file_id) and write the ids to download."""
    rows, ids = [], []
    for tumor, sig, *_ in pairs:
        tp_c, tn_c = pair_candidates(tumor, sig)
        for role, cs in [('TP', tp_c), ('TN', tn_c)]:
            if any(o['dl'] for o in cs): continue
            seen = set()
            for o in cs:
                if o['case'] in seen: continue
                seen.add(o['case']); rows.append((tumor, sig, role, o['case'], o['slide'], o['fid']))
                if o['fid'] not in ('?', 'ID_NON_TROVATO'): ids.append(o['fid'])
                if len(seen) >= alt: break
    ids = list(dict.fromkeys(ids))
    if not rows:
        print('Tutte le slide necessarie sono gia scaricate.'); return
    print('Slide da scaricare (piu alternative per ruolo; le "redacted" falliranno, ne basta una):')
    for r in rows: print('  %-6s %-8s %-3s %s  %s.svs  ->  %s' % r)
    with open(ids_file, 'w') as fh: fh.write('\n'.join(ids) + '\n')
    print(f'\n{len(ids)} id scritti in {ids_file}. Per scaricare:')
    print('  cd gdc && ./gdc-client download $(cat ids_paired.txt)')

### 5a. Scegli le coppie e scarica le slide mancanti

In [ ]:
# exemplars: 3 signatures of known etiology + 1 unknown, all with a strong signal and the TP already downloaded
PAIRS = [
    ('UCEC', 'SBS8'),
    ('LIHC', 'SBS12'),
    ('TGCT', 'SBS39'),
    ('READ', 'SBS18'),
]
download_plan(PAIRS)

### 5b. Genera le figure (dopo aver scaricato)

In [ ]:
for t, s, _ in PAIRS:
    print('==', t, s, '==')
    paired_tp_tn(t, s)

## 5c. TP only — no TN comparison
Same grid as `paired_tp_tn` but with a **single row**, the TP. No TN slide is needed.
`download_plan_tp_only` lists and downloads only the missing TP slides for the four pairs in `PAIRS`.


In [ ]:
def solo_tp(tumor, sig, K=PAIR_K, score=PAIR_SCORE):
    tp_c, _ = pair_candidates(tumor, sig)
    pick = lambda cs: next((o for o in cs if o['dl']), None)
    tp = pick(tp_c)
    if tp is None:
        print(f'[{tumor}/{sig}] nessuna TP scaricata -> usa download_plan_tp_only'); return
    wsi, thumb, sx, sy, factor, st, uniq, sel = row_data(tumor, sig, tp, K, score)
    fig = plt.figure(figsize=(9, 5.5))
    outer = gridspec.GridSpec(1, 2, width_ratios=[1.5, 1], wspace=0.06, figure=fig)
    axt = fig.add_subplot(outer[0, 0]); axt.imshow(thumb); axt.axis('off')
    for mask, col in [(uniq, '#1f77b4'), (~uniq, '#ff7f0e')]:
        s2 = st[mask]; axt.scatter(s2['coord_x']*factor*sx, s2['coord_y']*factor*sy, s=16, c=col, edgecolor='k', linewidth=0.3, zorder=3)
    for m, (_, pr) in enumerate(sel.iterrows(), 1):
        x, y = pr['coord_x']*factor*sx, pr['coord_y']*factor*sy
        axt.scatter([x], [y], s=120, facecolors='none', edgecolors='k', linewidths=1.6, zorder=4)
        axt.text(x, y, str(m), color='k', fontsize=9, fontweight='bold', ha='center', va='center', zorder=5)
    axt.set_title(f'TP  {tp["case"]}  U={float(uniq.mean()):.2f} (K={K})', fontsize=12)
    inner = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=outer[0, 1], wspace=0.08, hspace=0.22)
    for m, (_, pr) in enumerate(sel.iterrows()):
        axc = fig.add_subplot(inner[m//2, m%2]); im, _ = read_win(wsi, int(pr['coord_x']), int(pr['coord_y']), factor, CROP_CONTEXT)
        axc.imshow(im); axc.set_xticks([]); axc.set_yticks([])
        col = '#1f77b4' if pr['kind'] == 'unique' else '#ff7f0e'
        for sp in axc.spines.values(): sp.set_edgecolor(col); sp.set_linewidth(3)
        axc.set_title(f'#{m+1} {pr["kind"]}', fontsize=10, color=col)
    wsi.close()
    fig.legend(handles=[plt.Line2D([],[],marker='o',ls='',color='#1f77b4',label='unique',markersize=9),
                        plt.Line2D([],[],marker='o',ls='',color='#ff7f0e',label='shared',markersize=9)], loc='upper right', fontsize=10, frameon=False)
    fig.suptitle(f'{tumor} — {sig} (K={K}, {score}, vs co-dominant): solo TP', y=0.99, fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'soloTP_{tumor}_{sig}_{score}_K{K}.png'), dpi=200, bbox_inches='tight'); plt.show()

def download_plan_tp_only(pairs, alt=2, ids_file='../gdc/ids_tp_only.txt'):
    '''Like download_plan but TP role only: no TN slide required.'''
    rows, ids = [], []
    for tumor, sig, *_ in pairs:
        tp_c, _ = pair_candidates(tumor, sig)
        if any(o['dl'] for o in tp_c):
            continue
        seen = set()
        for o in tp_c:
            if o['case'] in seen: continue
            seen.add(o['case']); rows.append(('TP', tumor, sig, o['case'], o['slide'], o['fid']))
            if o['fid'] not in ('?', 'ID_NON_TROVATO'): ids.append(o['fid'])
            if len(seen) >= alt: break
    ids = list(dict.fromkeys(ids))
    if not rows:
        print('Tutte le TP necessarie sono gia scaricate.'); return
    print('TP da scaricare (piu alternative; ne basta una per coppia):')
    for r in rows: print('  %-3s %-6s %-8s %s  %s.svs  ->  %s' % r)
    with open(ids_file, 'w') as fh: fh.write('\n'.join(ids) + '\n')
    print(f'\n{len(ids)} id scritti in {ids_file}. Per scaricare:')
    print('  cd gdc && ./gdc-client download $(cat ids_tp_only.txt)')

### 5d. TP-only download plan and figures


In [ ]:
download_plan_tp_only(PAIRS)

In [ ]:
for t, s, *_ in PAIRS:
    print('==', t, s, '==')
    solo_tp(t, s)

## 6. Screening panels — READ/SBS18, UCEC/SBS8, THCA/SBS39 (K=20, TP only)
Browse the 18 downloaded slides to pick the best ones. One **row per slide**: thumbnail with the
top-K patches coloured (blue = unique, orange = shared with the co-dominant signatures) and the
first **unique** patches cropped and numbered on the thumbnail.

`PANEL_PER_FIG = 3` slides per figure keeps them legible; raise `PANEL_NCROP` for more crops.
The slide list and its order (decreasing U) come from `../gdc/u_selected_K20.csv`.


In [ ]:
U_SEL_CSV     = '../gdc/u_selected_K20.csv'
PANEL_K       = 20    # K per unique/shared
PANEL_PER_FIG = 3     # slide per figura (leggibilita')
PANEL_NCROP   = 4     # ritagli per slide
PANEL_CTX     = 2     # contesto attorno alla patch (1 = solo patch, 2-3 = piu' tessuto intorno)

def _panel_row_data(tumor, sig, case, slide, K, score):
    '''Like row_data but starting from (case, slide) instead of a pair_candidates dict.'''
    d = load_attr(tumor, case, slide)
    if d is None or d.empty:
        return None
    d = d[d['sample_id'].astype(str) == slide]
    svs = find_svs(slide)
    if svs is None:
        return None
    wsi = openslide.open_slide(svs); W, H = wsi.dimensions
    factor, _ = calibrate_factor(wsi, topk(d, sig, 6, score), slide_mag(wsi))
    thumb = wsi.get_thumbnail((THUMB_MAX, int(THUMB_MAX * H / W))); sx, sy = thumb.width / W, thumb.height / H
    others = others_of(d, sig, case)
    st = topk(d, sig, K, score)
    un = set().union(*[set(topk(d, l, K, score)['original_index']) for l in others]) if others else set()
    uniq = ~st['original_index'].isin(un)
    return dict(wsi=wsi, thumb=thumb, sx=sx, sy=sy, factor=factor, st=st, uniq=uniq,
                U=float(uniq.mean()), n_others=len(others))

def panel_slides(tumor, sig, K=PANEL_K, per_fig=PANEL_PER_FIG, n_crop=PANEL_NCROP,
                 score=None, ctx=PANEL_CTX, save=True):
    score = score or globals().get('PAIR_SCORE', 'combined')
    sel = pd.read_csv(U_SEL_CSV)
    sel = sel[(sel['tumor'].str.upper() == tumor.upper()) & (sel['signature'] == sig)]
    sel = sel.sort_values('U', ascending=False).reset_index(drop=True)
    todo = [r for _, r in sel.iterrows() if find_svs(r['slide']) is not None]
    missing = len(sel) - len(todo)
    print(f'[{tumor} {sig}] {len(todo)} slide scaricate' + (f' ({missing} mancanti, saltate)' if missing else ''))
    if not todo:
        return
    for start in range(0, len(todo), per_fig):
        chunk = todo[start:start + per_fig]
        fig = plt.figure(figsize=(5.0 + 2.5 * n_crop, 3.7 * len(chunk)))
        gs = gridspec.GridSpec(len(chunk), 1 + n_crop, width_ratios=[2.4] + [1] * n_crop,
                               figure=fig, wspace=0.06, hspace=0.30)
        for i, r in enumerate(chunk):
            rd = _panel_row_data(tumor, sig, r['case_id'], r['slide'], K, score)
            if rd is None:
                print(f'  [{r["case_id"]}] niente cache/slide -> skip'); continue
            axt = fig.add_subplot(gs[i, 0]); axt.imshow(rd['thumb']); axt.axis('off')
            st, uniq, factor, sx, sy = rd['st'], rd['uniq'], rd['factor'], rd['sx'], rd['sy']
            for mask, col in [(uniq, '#1f77b4'), (~uniq, '#ff7f0e')]:
                s2 = st[mask]
                axt.scatter(s2['coord_x'] * factor * sx, s2['coord_y'] * factor * sy,
                            s=18, c=col, edgecolor='k', linewidth=0.3, zorder=3)
            # crops: unique patches first (decreasing score), then shared ones if needed
            us = st[uniq].nlargest(n_crop, 'combined')
            n_u = len(us)
            if n_u < n_crop:
                us = pd.concat([us, st[~uniq].nlargest(n_crop - n_u, 'combined')])
            us = us.assign(kind=['unique'] * n_u + ['shared'] * (len(us) - n_u))
            for m, (_, pr) in enumerate(us.iterrows(), 1):
                x, y = pr['coord_x'] * factor * sx, pr['coord_y'] * factor * sy
                axt.scatter([x], [y], s=130, facecolors='none', edgecolors='k', linewidths=1.6, zorder=4)
                axt.text(x, y, str(m), color='k', fontsize=9, fontweight='bold',
                         ha='center', va='center', zorder=5)
            rank = start + i + 1
            axt.set_title(f'#{rank}  {r["case_id"]}   U={rd["U"]:.2f}  (co-dom={rd["n_others"]}, K={K})',
                          fontsize=11)
            for m, (_, pr) in enumerate(us.iterrows()):
                axc = fig.add_subplot(gs[i, 1 + m])
                im, _ = read_win(rd['wsi'], int(pr['coord_x']), int(pr['coord_y']), factor, ctx)
                axc.imshow(im); axc.set_xticks([]); axc.set_yticks([])
                col = '#1f77b4' if pr['kind'] == 'unique' else '#ff7f0e'
                for sp in axc.spines.values(): sp.set_edgecolor(col); sp.set_linewidth(3)
                axc.set_title(f'#{m+1} {pr["kind"]}', fontsize=9, color=col)
            rd['wsi'].close()
        fig.suptitle(f'{tumor} — {sig}  (K={K}, {score}, solo TP)   slide {start+1}-{start+len(chunk)} di {len(todo)}',
                     y=0.995, fontsize=13)
        if save:
            plt.savefig(os.path.join(FIG_DIR, f'panel_{tumor}_{sig}_K{K}_{start+1}-{start+len(chunk)}.png'),
                        dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
for tumor, sig in [('READ', 'SBS18'), ('UCEC', 'SBS8'), ('THCA', 'SBS39')]:
    print('=' * 60)
    panel_slides(tumor, sig)

## 8. Picking the candidate slides — U, tissue compactness, biology
Three tools, to be used in this order for each (tumor, signature) pair:

1. `u_rank(tumor, sig)` — every TP ranked by **U at K=20**, with the number of co-dominant
   signatures, how many separate fragments the WSI contains, and whether it is already
   downloaded. Discard cases with `n_others = 0`: there U = 1.00 by construction, not by signal.
2. Check `n_frammenti`: **1** means a compact thumbnail; 3-4 means split tissue, which looks bad
   in a figure.
3. `contact_sheet(tumor, sig, case)` — all top-20 patches, blue/orange borders. Use it to confirm
   that the **unique** patches sit on neoplastic epithelium and not on muscle, ink, the slide edge
   or normal tissue. This check is not optional: among the UCEC cases, `TCGA-A5-A2K2` had U = 0.40
   and a single fragment, yet all its unique patches fell on myometrium.


In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
import openslide, pyarrow.parquet as pq
from IPython.display import display

OUT_DIR_  = globals().get('OUT_DIR', '../grad/xai_outputs')
DL_DIR_   = globals().get('DL_DIR', '../gdc')
EXPL3_    = '../coad_stad_explainability/explain_top3.parquet'
FIG_DIR_  = os.path.join(OUT_DIR_, 'patch_figs'); os.makedirs(FIG_DIR_, exist_ok=True)
PATCH_20X_, K_DEF, COMBINED_ABS_ = 224, 20, True
_E3 = pq.read_table(EXPL3_).to_pandas()

def _svs(slide):
    h = glob.glob(os.path.join(DL_DIR_, '*', slide + '.svs')) + \
        glob.glob(os.path.join(DL_DIR_, slide + '.svs'))
    return h[0] if h else None

def _mag(w):
    for k in ('aperio.AppMag', openslide.PROPERTY_NAME_OBJECTIVE_POWER):
        v = w.properties.get(k)
        if v:
            try: return float(v)
            except ValueError: pass

def _factor(w):
    m = _mag(w)
    return max(1, int(round(m / 20))) if m else 2

def _win(w, x, y, f, ctx=1, disp=256):
    p0 = PATCH_20X_ * f; wn = p0 * ctx; W, H = w.dimensions
    ox = int(np.clip(x * f - (wn - p0) // 2, 0, max(0, W - wn)))
    oy = int(np.clip(y * f - (wn - p0) // 2, 0, max(0, H - wn)))
    return w.read_region((ox, oy), 0, (wn, wn)).convert('RGB').resize((disp, disp), Image.LANCZOS)

def frag_stats(thumb, thr=225):
    """(number of relevant fragments, share of the largest, bbox fill ratio)"""
    m = ndimage.binary_opening(np.asarray(thumb.convert('L')) < thr, np.ones((3, 3)))
    lab, n = ndimage.label(m)
    if n == 0: return 0, 0.0, 0.0
    sz = ndimage.sum(m, lab, range(1, n + 1)); tot = sz.sum()
    ys, xs = np.where(m)
    bb = (xs.max() - xs.min() + 1) * (ys.max() - ys.min() + 1)
    return int((sz > 0.05 * tot).sum()), float(sz.max() / tot), float(tot / bb)

def _attr(tumor, case, slide=None):
    d = pq.read_table(os.path.join(OUT_DIR_, f'attr_{tumor.lower()}_subset_scores.parquet'),
                      columns=['sample_id','case_id','signature_name','coord_x','coord_y',
                               'original_index','attention_weight','grad_score'],
                      filters=[('case_id', '==', case)]).to_pandas()
    if slide is not None:
        d = d[d['sample_id'].astype(str) == slide]
    d = d.copy()
    g = d['grad_score'].astype('float32')
    d['combined'] = d['attention_weight'].astype('float32') * (np.abs(g) if COMBINED_ABS_ else np.clip(g, 0, None))
    d['signature_name'] = d['signature_name'].astype(str)
    return d

def _codom(case, sig):
    s = set(_E3[(_E3['Case ID'] == case) & (_E3['group'] == 'plus')]['signature'].astype(str))
    return [x for x in s if x != sig]

def uniqueness(d_slide, sig, others, K=K_DEF):
    """Top-K patches of the signature plus a unique mask (absent from every co-dominant top-K)."""
    st = d_slide[d_slide['signature_name'] == sig].nlargest(K, 'combined')
    un = set()
    for o in others:
        un |= set(d_slide[d_slide['signature_name'] == o].nlargest(K, 'combined')['original_index'])
    return st, ~st['original_index'].isin(un)

def u_rank(tumor, sig, K=K_DEF, only_downloaded=False, with_frag=True):
    """TPs of (tumor, signature) ranked by U, with WSI compactness and download status."""
    es = _E3[(_E3['tumor_type'].str.upper() == tumor.upper()) & (_E3['signature'] == sig)]
    tp = [r['Case ID'] for _, r in es.iterrows() if r['group'] == 'plus' and r['eligible'] == 'yes']
    rows = []
    for c in tp:
        try:
            d = _attr(tumor, c)
        except Exception:
            continue
        if d.empty: continue
        others = _codom(c, sig)
        for slide in sorted(d['sample_id'].astype(str).unique()):
            ds = d[d['sample_id'].astype(str) == slide]
            st, uniq = uniqueness(ds, sig, others, K)
            if st.empty: continue
            p = _svs(slide)
            row = dict(case_id=c, slide=slide, n_others=len(others),
                       U=round(float(uniq.mean()), 3), scaricata=p is not None)
            if with_frag and p is not None:
                w = openslide.open_slide(p); W, H = w.dimensions
                nf, lg, fl = frag_stats(w.get_thumbnail((1200, int(1200 * H / W))))
                w.close()
                row.update(n_frammenti=nf, frammento_max=round(lg, 2), riempimento=round(fl, 2))
            else:
                row.update(n_frammenti=np.nan, frammento_max=np.nan, riempimento=np.nan)
            rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty: return df
    if only_downloaded: df = df[df.scaricata]
    return (df.sort_values('U', ascending=False)
              .drop_duplicates('case_id')
              .reset_index(drop=True))

def contact_sheet(tumor, sig, case, slide=None, K=K_DEF, save=True):
    """All top-K patches: blue border = unique, orange = shared. Used to validate the biology."""
    d = _attr(tumor, case)
    if d.empty: print('nessuna attribuzione in cache'); return
    slide = slide or sorted(d['sample_id'].astype(str).unique())[0]
    ds = d[d['sample_id'].astype(str) == slide]
    others = _codom(case, sig)
    st, uniq = uniqueness(ds, sig, others, K)
    st = st.reset_index(drop=True); uniq = uniq.reset_index(drop=True)
    p = _svs(slide)
    if p is None: print(f'{slide}.svs non scaricata'); return
    w = openslide.open_slide(p); f = _factor(w)
    ncol = 5; nrow = int(np.ceil(len(st) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.7 * nrow))
    for ax, (i, pr) in zip(np.atleast_1d(axes).ravel(), st.iterrows()):
        ax.imshow(_win(w, int(pr['coord_x']), int(pr['coord_y']), f))
        ax.set_xticks([]); ax.set_yticks([])
        kind = 'unique' if uniq[i] else 'shared'
        c = '#2a78d6' if uniq[i] else '#eb6834'
        for sp in ax.spines.values(): sp.set_edgecolor(c); sp.set_linewidth(3)
        ax.set_title(f'#{i+1} {kind}', fontsize=9, color=c)
    for ax in np.atleast_1d(axes).ravel()[len(st):]: ax.axis('off')
    fig.suptitle(f'{tumor} · {sig} · {case}   U={float(uniq.mean()):.2f}  co-dom={len(others)}  (top-{K})',
                 fontsize=13)
    w.close(); plt.tight_layout()
    if save:
        plt.savefig(os.path.join(FIG_DIR_, f'contact_{tumor}_{sig}_{case}.png'), dpi=110, bbox_inches='tight')
    plt.show()

# --- usage: LIHC / SBS12 (the fourth panel) --------------------------------------
lihc = u_rank('LIHC', 'SBS12')
print('LIHC/SBS12 — scarta n_others=0; cerca U alto + n_frammenti=1')
display(lihc[lihc.n_others > 0].head(15))


## 9. Page 1 — signatures of **unknown** etiology with a spatial signal
**Run section 8 first**: it defines `u_rank`, `contact_sheet`, `_attr` and `uniqueness`.

One figure per (tumor, signature): slide overview with the signature's top-K patches (filled
squares, a single colour, no unique/shared split) and the three highest-scoring crops. This cell
also defines `make_figures`, reused by the page 2 cell.

`case=None` picks automatically among the downloaded slides that have single-fragment tissue and
at least one co-dominant signature. **Always confirm with `contact_sheet` before trusting the
automatic pick**: a high U and a single fragment do not guarantee the patches are on tumor.


In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import openslide

NCROP, CTX_FIG, DISPLAY_FIG, THUMB_FIG = 3, 2, 640, 1600
C_MARK, INK, INK2 = '#2a78d6', '#0b0b0b', '#52514e'
SAVE_DIR = '.'                      # le figure escono nella cartella dei notebook

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'normal', 'axes.titleweight': 'normal', 'axes.labelweight': 'normal',
    'text.color': INK, 'axes.edgecolor': INK2, 'figure.facecolor': 'white',
    'savefig.facecolor': 'white', 'pdf.fonttype': 42, 'ps.fonttype': 42,
})

def _autopick(tumor, sig, K):
    t = u_rank(tumor, sig, K, only_downloaded=True)
    if t.empty:
        raise RuntimeError(f'{tumor}/{sig}: nessuna slide scaricata')
    ok = t[(t.n_others > 0) & (t.n_frammenti == 1)]
    if ok.empty:
        ok = t[t.n_others > 0]
        print(f'  [{tumor}/{sig}] nessuna slide a frammento unico: uso la migliore comunque')
    r = ok.iloc[0]
    print(f'  [{tumor}/{sig}] auto -> {r.case_id}  U={r.U}  co-dom={r.n_others}  frammenti={r.n_frammenti}')
    return r.case_id, r.slide

def tissue_bbox(thumb, thr=225, frac=0.002, pad=0.02):
    a = np.asarray(thumb.convert('L')); m = a < thr; H, W = m.shape
    cols = np.where(m.sum(0) > max(2, frac * H))[0]
    rows = np.where(m.sum(1) > max(2, frac * W))[0]
    if len(cols) == 0 or len(rows) == 0:
        return 0, 0, W, H
    x0, x1, y0, y1 = cols[0], cols[-1] + 1, rows[0], rows[-1] + 1
    px, py = int(pad * (x1 - x0)), int(pad * (y1 - y0))
    return max(0, x0 - px), max(0, y0 - py), min(W, x1 + px), min(H, y1 + py)

def _read_win(w, x, y, f, ctx):
    p0 = PATCH_20X_ * f; wn = p0 * ctx; W, H = w.dimensions
    ox = int(np.clip(x * f - (wn - p0) // 2, 0, max(0, W - wn)))
    oy = int(np.clip(y * f - (wn - p0) // 2, 0, max(0, H - wn)))
    return w.read_region((ox, oy), 0, (wn, wn)).convert('RGB').resize((DISPLAY_FIG, DISPLAY_FIG), Image.LANCZOS)

def _calib(w, top):
    W, H = w.dimensions; m = _mag(w)
    f0 = max(1, int(round(m / 20))) if m else None
    cand = ([f0] + [f for f in (2, 1) if f != f0]) if f0 else [2, 1]
    sc = {}
    for f in cand:
        wf, n = 0.0, 0
        for _, pr in top.head(6).iterrows():
            x, y = int(pr['coord_x']), int(pr['coord_y'])
            if x * f + PATCH_20X_ * f > W or y * f + PATCH_20X_ * f > H:
                wf, n = 1.0, 1; break
            wf += float((np.asarray(_read_win(w, x, y, f, 1).convert('L')) > 220).mean()); n += 1
        sc[f] = wf / max(n, 1)
    return min(sc, key=sc.get)

def _row(fig, sub_gs, letter, tumor, sig, case, slide, K):
    """letter=None -> no panel letter, captions flush with the image edge."""
    """One row: slide overview plus NCROP crops. Returns a record of what it used."""
    inner = gridspec.GridSpecFromSubplotSpec(1, 1 + NCROP, subplot_spec=sub_gs,
                                             width_ratios=[2.25] + [1] * NCROP, wspace=0.035)
    d = _attr(tumor, case)
    slide = slide or sorted(d['sample_id'].astype(str).unique())[0]
    d = d[d['sample_id'].astype(str) == slide]
    others = _codom(case, sig)
    st, uniq = uniqueness(d, sig, others, K)
    U = float(uniq.mean())

    w = openslide.open_slide(_svs(slide)); W, H = w.dimensions
    factor = _calib(w, st)
    thumb = w.get_thumbnail((THUMB_FIG, int(THUMB_FIG * H / W)))
    sx, sy = thumb.width / W, thumb.height / H
    bx0, by0, bx1, by1 = tissue_bbox(thumb)
    thumb = thumb.crop((bx0, by0, bx1, by1))

    axt = fig.add_subplot(inner[0]); axt.imshow(thumb); axt.set_xticks([]); axt.set_yticks([])
    # imshow fixes the axes aspect and centres them in the grid cell. Thumbnails have
    # different aspect ratios, so without this every row would start at a different x
    # and the captions, anchored to the axes, would come out misaligned.
    axt.set_anchor('W')
    for sp in axt.spines.values():
        sp.set_edgecolor('#d8d7d2'); sp.set_linewidth(0.8)
    # only the signature's top-K, a single colour, filled squares with no halo
    axt.scatter(st['coord_x'] * factor * sx - bx0, st['coord_y'] * factor * sy - by0,
                s=42, c=C_MARK, marker='s', linewidths=0, zorder=4)

    # crops: highest-scoring patches, skipping those that are mostly background
    cand = st.sort_values('combined', ascending=False)
    keep = []
    for i, pr in cand.iterrows():
        im = _read_win(w, int(pr['coord_x']), int(pr['coord_y']), factor, CTX_FIG)
        if float((np.asarray(im.convert('L')) > 225).mean()) < 0.45:
            keep.append(i)
        if len(keep) == NCROP:
            break
    picks = cand.loc[keep] if len(keep) >= NCROP else cand.head(NCROP)

    for m, (_, pr) in enumerate(picks.iterrows(), 1):
        px = pr['coord_x'] * factor * sx - bx0
        py = pr['coord_y'] * factor * sy - by0
        axt.scatter([px], [py], s=200, c=C_MARK, marker='s', linewidths=0, zorder=6)
        axt.text(px, py, str(m), color='white', fontsize=8.5, ha='center', va='center', zorder=7)

    dx = 20 if letter else 0
    if letter:
        axt.annotate(letter, xy=(0, 1), xycoords='axes fraction', xytext=(0, 30),
                     textcoords='offset points', fontsize=14, color=INK, va='bottom', ha='left')
    axt.annotate(f'{tumor} · {sig}', xy=(0, 1), xycoords='axes fraction', xytext=(dx, 30),
                 textcoords='offset points', fontsize=13, color=INK, va='bottom', ha='left')
    axt.annotate(f'{case}   ·   U = {U:.2f}   ·   top-{K}', xy=(0, 1), xycoords='axes fraction',
                 xytext=(dx, 12), textcoords='offset points', fontsize=9.5, color=INK2,
                 va='bottom', ha='left')

    for m, (_, pr) in enumerate(picks.iterrows(), 1):
        axc = fig.add_subplot(inner[m])
        axc.imshow(_read_win(w, int(pr['coord_x']), int(pr['coord_y']), factor, CTX_FIG))
        axc.set_xticks([]); axc.set_yticks([]); axc.set_anchor('W')
        for sp in axc.spines.values():
            sp.set_edgecolor(C_MARK); sp.set_linewidth(2.6)
        axc.text(0.06, 0.94, str(m), transform=axc.transAxes, fontsize=10, color='white',
                 ha='center', va='center', zorder=8,
                 bbox=dict(boxstyle='square,pad=0.35', fc=C_MARK, ec='none'))
    w.close()
    return dict(tumor=tumor, sig=sig, case=case, slide=slide, U=round(U, 3), n_others=len(others), K=K)

def make_figures(rows, K, save_dir=SAVE_DIR):
    """One separate PNG per entry in `rows`. No panel letter.
    File name: fig_<TUMOR>_<SIGNATURE>.png. The left margin is identical across figures
    (set_anchor('W') plus flush captions), so they stay aligned with each other."""
    out = []
    for r in rows:
        case, slide = (r['case'], r.get('slide')) if r.get('case') else _autopick(r['tumor'], r['sig'], K)
        fig = plt.figure(figsize=(12.4, 3.3))
        gs = gridspec.GridSpec(1, 1, figure=fig, top=0.80, bottom=0.02, left=0.035, right=0.985)
        info = _row(fig, gs[0], None, r['tumor'], r['sig'], case, slide, K)
        stem = os.path.join(save_dir, f"fig_{r['tumor']}_{r['sig']}")
        fig.savefig(stem + '.png', dpi=400, bbox_inches='tight')
        plt.show()
        print(f'  -> {stem}.png')
        out.append(info)
    return pd.DataFrame(out)

# ---------------- PAGE 1: unknown etiology ----------------
K_PAGE1 = 20        # SBS8/SBS12/SBS39 significative a K=20; SBS17b lo e' sia a K=10 sia a K=20
PAGE1 = [
    dict(tumor='UCEC', sig='SBS8',   case='TCGA-AJ-A3BI'),
    dict(tumor='LIHC', sig='SBS12',  case='TCGA-DD-AACE'),
    dict(tumor='THCA', sig='SBS39',  case='TCGA-ET-A25O'),
    dict(tumor='STAD', sig='SBS17b', case=None),      # nuova: auto fra le 6 scaricate
]
display(make_figures(PAGE1, K_PAGE1))


## 10. Page 2 — signatures of **established** etiology
Same layout, reusing `make_figures` from the previous cell, so run that one first.

**On the choice of K.** On the `jaccard` metric — the one on which these pairs come out
significant in `jaccard_uniqueness_ALL_top5_absC.csv` — only CESC/SBS2 (p = 0.0026) and
ESCA/SBS13 (p = 0.018) pass at K = 10. COAD/SBS44 and READ/SBS18 become significant from K = 20
upwards (p = 0.0038 and p = 0.0095) and improve further at K = 50/100. `K_PAGE2 = 20` keeps all
four; set it to 10 only if you accept that two panels remain visual examples without statistical
support.


In [ ]:
# ---------------- PAGE 2: established etiology ----------------
K_PAGE2 = 20        # 10 = come chiesto, ma SBS44 e SBS18 non sono significative a quel K (vedi sopra)
PAGE2 = [
    dict(tumor='READ', sig='SBS18', case='TCGA-DC-5337'),   # gia' validata visivamente
    dict(tumor='COAD', sig='SBS44', case=None),
    # chosen from the contact sheets: the highest-scoring patches fall on carcinoma.
    # The earlier automatic picks (BI-A20A, VR-A8EW) had higher U but attributed to
    # stroma/smooth muscle and to mature, non-neoplastic squamous epithelium.
    dict(tumor='CESC', sig='SBS2',  case='TCGA-C5-A7UI'),
    dict(tumor='ESCA', sig='SBS13', case='TCGA-VR-AA4G'),
]
display(make_figures(PAGE2, K_PAGE2))
